# Create the per cell type pseudobulk matrix

We create a pseudobulk matrix (aggregates cell counts by sample) for each `cell_type`, and for each `main_cell_type`
As a minimal working example (MWE), it is perform like that for each cell type: 

In [ ]:
library(Seurat)
astro<-readRDS('../outputs/01-SEAAD_data/DLPFC/Astrocyte.rds')
pseudo_mat<-AggregateExpression(astro,assays = 'RNA',slot = 'count',group.by = 'Donor.ID',
                    return.seurat = FALSE)
fwrite(data.table(pseudo_mat$RNA,keep.rownames = 'gene_id'),'outputs/01-SEAAD_data/DLPFC/Astrocyte_pseudobulk.csv.gz')



The script for all cell type and brain region are in ['scripts/01Dii-create_pseudobulk_cell_type.R']('../scripts/01Dii-create_pseudobulk_cell_type.R')

### merge the splitted cell type into one pseudobulk data
for the cell type divided into different object/rds file(because too large data),we aggregate expression into one.  
we thus create a data.table `pseudo_files_dt` saving these information

In [ ]:
region='DLPFC'
out1<-fp(out,region)
mtd<-fread(fp(out1,'all_final_RNAseq_nuclei_metadata.csv.gz'))

#to match with the rds object name, we create a column containing the original cell type name 
mtd[,original_cell_type_name:=str_remove(str_replace_all(cell_type,' ','_'),'^Inh_|Exc_')]
mtd[,original_cell_type_name:=str_replace_all(original_cell_type_name,'/','-')]
fwrite(mtd,fp(out1,'all_final_RNAseq_nuclei_metadata.csv.gz')) 
  
mtsc<-unique(mtd,by=c('Donor.ID','cell_type'))
  
pseudo_files<-list.files(out1,pattern = '\\_pseudobulk\\.csv\\.gz',full.names = T)
pseudo_files_dt<-data.table(file=pseudo_files,
                              original_cell_type_name=str_remove(basename(pseudo_files),'_pseudobulk\\.csv\\.gz'))
pseudo_files_dt[,original_cell_type_name:=str_remove(original_cell_type_name,'\\_set[0-9]+')]
  
  
pseudo_files_dt<-merge(pseudo_files_dt,unique(mtsc[,.(original_cell_type_name,cell_type,main_cell_type)]))
pseudo_files_dt[,n_file:=.N,by='cell_type']
  
for(ct in unique(pseudo_files_dt[n_file>1]$original_cell_type_name)){
    pseudo_list<-lapply(pseudo_files_dt[original_cell_type_name==ct]$file, function(f)fread(f))
    
    #lacking samples column
    samples<-Reduce(union,lapply(pseudo_list,colnames))
    pseudo_list<-lapply(pseudo_list,function(x){
      samples_lacking<-setdiff(samples,colnames(x))
      x[,(samples_lacking):=0]
    })
    
    #transform as matrix
    pseudo_list<-lapply(pseudo_list, function(x)as.matrix(data.frame(x,row.names = 'gene_id')))
    
    #aggregate count by sample / featute
    features<-rownames(pseudo_list[[1]])
    samples<-colnames(pseudo_list[[1]])
    
    pseudo_merge<-Reduce(`+`,lapply(pseudo_list,function(x)x[features,samples]))
    print(head(pseudo_merge[,1:10]))
    fwrite(data.table(pseudo_merge,keep.rownames = 'gene_id'),fp(out1,paste0(ct,'pseudobulk.csv.gz')))
    
  }
                                    
  
#we do not need anymore of the pseudobulk data from the divided cell type objects, so we remove them
system(paste('rm',paste(pseudo_files_dt[n_file>1]$file,collapse = ' ')))
  
  

###  Create metadata for each pseudobulk 
In addition to the clinical covariates, we also store in each cell type level metadata different QC metrics like the proportion of this cell type in the sample, the median UMIs count and median gene detectedm as well as average %mitochondrial count that we could used as covariated in our pseudobulk analysis.  
Importantly, we also flag here samples with **less than 50 cells** for this cell type, This is a flag that we should use to exclude sample with not enough cells to overcome drop out events.

In [ ]:
#save diverse cell type level info 
mtd[,tot.cells.donor:=.N,by=.(`Donor.ID`)]
mtd[,n.cells:=.N,by=.(`Donor.ID`,cell_type)]

mtd[,prop.cells:=n.cells/tot.cells.donor,by=.(`Donor.ID`,cell_type)]
mtd[,med.umis.per.cell:=median(Number.of.UMIs,na.rm = T),by=.(`Donor.ID`,cell_type)]
mtd[,med.genes.per.cell:=median(Genes.detected,na.rm = T),by=.(`Donor.ID`,cell_type)]
mtd[,avg.pct.mt.per.cell:=mean(`Fraction.mitochondrial.UMIs`),by=c('Donor.ID','cell_type')]
mtsc<-unique(mtd,by=c('Donor.ID','cell_type'))
  
#flag donors with not enough cells
mtd[,pass.threshold.n.cells:=n.cells>50]
mtd[,outlier.n.cells:=!pass.threshold.n.cells,by=c('Donor.ID','cell_type')]
  
mtscf<-RemoveUselessColumns(mtsc,key_cols=c('Donor.ID','cell_type'),pattern_to_exclude = 'ATAC|Multiome|Doublet|Number.of|Genes.')
  
fwrite(mtscf,fp(out1,'all_final_RNAseq_nuclei_cell_type_level_metadata.csv.gz'))


## Pseudobulk by Main Cell type<a class="anchor" id="7"></a>
We then aggregated the pseudobulk matrix of each cell type belonging to the same lower resolution cell type annoation (called `main_cell_type` in metadata). e.g, we sum counts for all Excitatory neurons type together.
  
We also create the metadata associated to it  similarly than for the more resolutive level.


In [ ]:

 # merge by main_cell_type
out2<-fp(out1,'pseudobulk_main_cell_type')
dir.create(out2)
pseudo_files<-list.files(out1,pattern = '\\_pseudobulk\\.csv\\.gz',full.names = T)
to_merge<-data.table(file=pseudo_files,
                       original_cell_type_name=str_remove(basename(pseudo_files),'_pseudobulk\\.csv\\.gz'))
  
  
to_merge<-merge(to_merge,unique(mtsc[,.(original_cell_type_name,cell_type,main_cell_type)]))
  
for(ct in unique(to_merge$main_cell_type)){
    message(ct)
    pseudo_list<-lapply(to_merge[main_cell_type==ct]$file, function(f)fread(f))
    
    #lacking samples column
    samples<-Reduce(union,lapply(pseudo_list,colnames))
    pseudo_list<-lapply(pseudo_list,function(x){
      samples_lacking<-setdiff(samples,colnames(x))
      x[,(samples_lacking):=0]
    })
    
    #transform as matrix
    pseudo_list<-lapply(pseudo_list, function(x)as.matrix(data.frame(x,row.names = 'gene_id')))
    
    #aggregate count by sample / featute
    features<-rownames(pseudo_list[[1]])
    samples<-colnames(pseudo_list[[1]])
    
    pseudo_merge<-Reduce(`+`,lapply(pseudo_list,function(x)x[features,samples]))
    print(head(pseudo_merge[,1:10]))
    fwrite(data.table(pseudo_merge,keep.rownames = 'gene_id'),fp(out2,paste0(ct,'.csv.gz')))
    
    
    #4) create mtd for each pseudobulk main cell type 

    mtd[,tot.cells.donor:=.N,by=.(`Donor.ID`)]
    mtd[,n.cells:=.N,by=.(`Donor.ID`,main_cell_type)]
    
    mtd[,prop.cells:=n.cells/tot.cells.donor,by=.(`Donor.ID`,main_cell_type)]
    
    mtd[,med.umis.per.cell:=median(Number.of.UMIs,na.rm = T),by=.(`Donor.ID`,main_cell_type)]
    mtd[,med.genes.per.cell:=median(Genes.detected,na.rm = T),by=.(`Donor.ID`,main_cell_type)]
    
    mtd[,avg.pct.mt.per.cell:=mean(`Fraction.mitochondrial.UMIs`),by=c('Donor.ID','main_cell_type')]
    
    mtsc<-unique(mtd,by=c('Donor.ID','main_cell_type'))
    
    #flag donors with not enough cells
    mtd[,pass.threshold.n.cells:=n.cells>50,by=]
    mtd[,outlier.n.cells:=!pass.threshold.n.cells,by=c('Donor.ID','main_cell_type')]
    
    mtscf<-RemoveUselessColumns(mtsc,key_cols=c('Donor.ID','main_cell_type'),pattern_to_exclude = 'ATAC|Multiome|Doublet|Number.of|Genes.')
    
    
    fwrite(mtscf,fp(out2,'all_final_RNAseq_nuclei_main_cell_type_level_metadata.csv.gz'))
    
  }

We saved everything at the same place in `outputs/01-SEAAD_data/[BrainRegion]/pseudobulk_main_cell_type`

## Next Step
Now that we have the pseudobulk data, we can perform differential exxpression analyis. see [this notebook](Pseudobulk_DESeq2_analysis_of_SEAAD_data.ipynb)